# 一个完成的训练过程

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

我们将了解如何在**不使用 Trainer 类的情况**下实现与上一节相同的结果。

假设你已经完成了第 2 节中的数据处理如下：

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    # truncation=True 截断超长文本
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

##训练前的准备

我们需要定义一些对象。但在定义这些数据加载器之前，我们需要对我们的 `tokenized_datasets` 进行一些**后处理，以自己实现一些 Trainer 自动为我们处理的内容**。

*  `.remove_columns`：删除与模型不需要的列（如 sentence1 和 sentence2 列）。
*  `.rename_column`：将列名 label 重命名为 labels （因为模型默认的输入是 labels ）。
*  `.set_format`：设置数据集的格式，使其返回 PyTorch 张量而不是列表。

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

检查结果中是否只有模型能够接受的列：

In [ ]:
["attention_mask", "input_ids", "labels", "token_type_ids"]

至此，我们可以轻松**定义数据加载器**：

In [ ]:
from torch.utils.data import DataLoader

# collate_fn=data_collator
# 默认会直接对样本列表执行 torch.stack，要求所有样本张量长度完全一致。但文本经过 tokenizer 后句子长度各不相同，直接 stack 会报错。
# 不要用默认堆叠方式，交给data_collator完成动态 padding + 组装 batch 张量。

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

为了快速检验数据处理中没有错误，我们可以这样检验其中的一个 batch：

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'attention_mask': torch.Size([8, 65]),
 'input_ids': torch.Size([8, 65]),
 'labels': torch.Size([8]),
 'token_type_ids': torch.Size([8, 65])}

现在我们已经完全完成了数据预处理，让我们将注意力转向模型。我们会像在上一节中所做的那样实例化它：

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

为了确保训练过程中一切顺利，我们将 batch 传递给这个模型：

In [ ]:
# batch是字典，，一个批次数据
# key：input_ids、attention_mask、token_type_ids、labels
# value：张量
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.5441, grad_fn=<NllLossBackward>) torch.Size([8, 2])

当我们输入 labels 时，🤗 Transformers 模型都将返回这个 batch 的 loss ，我们还得到了 logits （ batch 中的每个输入有两个输出，所以张量大小为 8 x 2）。


我们只是缺少优化器和学习率调度器。

由于我们试图手动实现 Trainer 的功能，我们将使用相同的优化器和学习率调度器。 Trainer 使用的优化器是 AdamW ，它与 Adam 相同，但加入了权重衰减正则化的一点变化。

In [ ]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

最后，默认使用的学习率调度器只是从最大值 （5e-5） 到 0 的线性衰减。为了定义它，我们需要知道我们训练的次数，即所有数据训练的次数（epochs）乘以的 batch 的数量（即我们训练数据加载器的长度）。 Trainer 默认情况下使用三个 epochs ，因此我们定义训练过程如下：

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377

##训练循环

我们定义了一个 device ，它在 GPU 可用的情况下指向 GPU，最后我们将把我们的模型和 batch 放在 device 上：

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

为了知道训练何时结束，我们使用 tqdm 库，在训练步骤数上添加了一个进度条

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

训练循环与上一节类似，我们没有要求在训练的过程中进行检验，所以这个训练循环不会告诉我们任何关于模型目前的状态。我们需要为此添加一个评估循环。

##评估循环

正如我们之前所做的那样，我们将使用 🤗 Evaluate 库提供的指标。

我们已经了解了 `metric.compute()` 方法，当我们使用 `add_batch()` 方法进行预测循环时，实际上该指标可以为我们累积所有 batch 的结果。一旦我们累积了所有 batch ，我们就可以使用 metric.compute() 评估得到的结果。

In [ ]:
import evaluate

# 加载 GLUE 基准下 MRPC 任务配套指标（准确率 Accuracy + F1 分数），metric 内部会缓存预测与标签。
metric = evaluate.load("glue", "mrpc")
# 模型评估模式
# model.eval()：关闭 Dropout、BatchNorm 训练行为；不启用梯度计算。
model.eval()
for batch in eval_dataloader:
    # 把张量搬运到 GPU / 对应设备
    batch = {k: v.to(device) for k, v in batch.items()}
    # 禁用梯度计算，节省显存、加快推理。
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    # 不立刻计算指标，逐批次缓存预测值与真实标签，适合大数据集，避免一次性加载全部预测。
    metric.add_batch(predictions=predictions, references=batch["labels"])

# 所有 batch 遍历完成后，一次性汇总全部数据，输出 accuracy、F1。
metric.compute()

{'accuracy': 0.8431372549019608, 'f1': 0.8907849829351535}

##使用 Accelerate 加速你的训练循环

只需进行一些调整，**我们就可以在多个 GPU 或 TPU 上启用分布式训练。**

从创建训练和验证数据加载器开始，我们的手动训练循环如下所示：

In [ ]:
from transformers import AdamW, AutoModelForSequenceClassification, get_scheduler

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

以下是更改后的版本：

In [ ]:
# new
from accelerate import Accelerator
from transformers import AdamW, AutoModelForSequenceClassification, get_scheduler
# new，实例化一个 Accelerator 对象 它将查看环境并初始化适当的分布式设置
accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

#Accelerate 为你处理数据在设备间的数据传递，因此你可以删除将模型放在设备上的那行代码
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# model.to(device)


# new
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        # batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        # loss.backward()
        # new
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

把这个放在 train.py 文件中，可以让它在任何类型的分布式设置上运行。要在分布式设置中试用它，请运行以下命令：


```
accelerate config
```
这将询问你几个配置的问题并将你的回答保存到此命令使用的配置文件中：

```
accelerate launch train.py
```
这将启动分布式训练

如果你想在 Notebook 中尝试此操作（例如，在 Colab 上使用 TPU 进行测试），只需将代码粘贴到一个 training_function() 函数中，并在最后一个单元格中运行：

In [ ]:
from accelerate import notebook_launcher

notebook_launcher(training_function)